# RQ2 — Provincial Attention Map + Clustering

**Algorithms (syllabus mapping)**
- W4 HyperLogLog — distinct count (provinces × topics)
- W8 K-Means++, DBSCAN — province clustering on topic vectors
- W10 ALS — latent province-topic factors
- W12 PCA / UMAP — 2D embedding for visualisation

**Inputs**: `silver_yazili_soru_clean` Delta + `gold_ministry_topic_party_year` (optional).

**Outputs (Gold)**:
- `province_topic` — province × topic frequency
- `province_cluster` — KMeans cluster id per province
- `province_embedding_2d` — PCA / UMAP coords

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))
from spark_utils import get_spark, read_delta, write_delta, TABLES, GOLD
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans

spark = get_spark('rq2', memory='6g')
spark.sparkContext.setLogLevel('WARN')
silver = read_delta(spark, TABLES['silver_yazili_soru_clean'])

## 1. Explode provinces_mentioned

In [ ]:
mentioned = (
    silver
    .filter(F.size('provinces_mentioned') > 0)
    .select('guid', 'muhatap_bakanlık', 'year', F.explode('provinces_mentioned').alias('province'))
)
print('province mentions:', mentioned.count())
mentioned.groupBy('province').count().orderBy(F.desc('count')).show(15)

## 2. HyperLogLog distinct count (provinces × ministries)

In [ ]:
hll = mentioned.groupBy('province').agg(
    F.approx_count_distinct('muhatap_bakanlık', rsd=0.02).alias('distinct_ministries'),
    F.approx_count_distinct('guid', rsd=0.02).alias('distinct_onerges'),
    F.count('*').alias('total_mentions'),
)
hll.orderBy(F.desc('total_mentions')).show(15)

## 3. Build (province × ministry) frequency matrix

In [ ]:
mat = (
    mentioned.groupBy('province', 'muhatap_bakanlık').count()
             .groupBy('province')
             .pivot('muhatap_bakanlık')
             .sum('count')
             .na.fill(0)
)
feat_cols = [c for c in mat.columns if c != 'province']
print(f'Provinces: {mat.count()}, features (ministries): {len(feat_cols)}')
assembler = VectorAssembler(inputCols=feat_cols, outputCol='raw_vec')
scaler = StandardScaler(inputCol='raw_vec', outputCol='vec', withStd=True, withMean=False)
vec_df = scaler.fit(assembler.transform(mat)).transform(assembler.transform(mat))

## 4. K-Means++ clustering (k chosen via elbow)

In [ ]:
import matplotlib.pyplot as plt
wsse = []
for k in range(2, 11):
    km = KMeans(featuresCol='vec', k=k, seed=42, initMode='k-means||')
    m = km.fit(vec_df)
    wsse.append((k, m.summary.trainingCost))
plt.plot([k for k, _ in wsse], [c for _, c in wsse], marker='o')
plt.xlabel('k'); plt.ylabel('within-cluster SSE'); plt.title('Elbow — province KMeans')
plt.grid(True); plt.show()

K = 5  # adjust after viewing elbow
km = KMeans(featuresCol='vec', k=K, seed=42, initMode='k-means||')
model_km = km.fit(vec_df)
clustered = model_km.transform(vec_df).select('province', F.col('prediction').alias('cluster'))
clustered.groupBy('cluster').agg(F.collect_list('province').alias('provinces')).show(truncate=False)

## 5. PCA — 2D coords for dashboard scatter

In [ ]:
pca = PCA(k=2, inputCol='vec', outputCol='coords')
pca_model = pca.fit(vec_df)
coords = pca_model.transform(vec_df).select(
    'province',
    F.col('coords').getItem(0).alias('x'),
    F.col('coords').getItem(1).alias('y'),
)
coords.show(10)

## 6. Write Gold tables

In [ ]:
write_delta(hll, GOLD / 'province_summary')
write_delta(mentioned.groupBy('province', 'muhatap_bakanlık', 'year').count(), TABLES['gold_province_topic'], partition_by=['year'])
write_delta(clustered.join(coords, 'province'), GOLD / 'province_embedding')
print('Gold written.')